In [1]:
import time
import random
import requests
from bs4 import BeautifulSoup
from langdetect import detect
import json

In [4]:
filmes_para_linkar = {
    "O corpo": "https://letterboxd.com/film/body-2015-3/",  # Ano: 2015
    "O Teto Sobre Nós": "https://letterboxd.com/film/the-roof-above-us/",  # Ano: 2015
    "Bá": "https://letterboxd.com/film/ba/",  # Ano: 2015
    "Miss & Grubs": "https://letterboxd.com/film/miss-grubs/",  # Ano: 2015
    "Rosinha": "https://letterboxd.com/film/rosinha/",  # Ano: 2016
    "Tailor": "https://letterboxd.com/film/tailor/",  # Ano: 2017
    "O Espirito do Bosque": "https://letterboxd.com/film/the-spirit-of-the-woods/",  # Ano: 2017
    "Telentrega": "https://letterboxd.com/film/telentrega/",  # Ano: 2017
    "O Violeiro Fantasma": "https://letterboxd.com/film/o-violeiro-fantasma/",  # Ano: 2017
    "O Quebra-Cabeça de Sara": "https://letterboxd.com/film/saras-puzzle/",  # Ano: 2017
    "Cabelo Bom": "https://letterboxd.com/film/curly-power/",  # Ano: 2017
    "Aquarela": "https://letterboxd.com/film/drawing/",  # Ano: 2018
    "Kairo": "https://letterboxd.com/film/kairo/",  # Ano: 2018
    "Menino Pássaro": "https://letterboxd.com/film/bird-boy-2018/",  # Ano: 2018
    "A Mulher que Sou": "https://letterboxd.com/film/a-mulher-que-sou/",  # Ano: 2019
    "O Véu de Amani": "https://letterboxd.com/film/o-veu-de-amani/",  # Ano: 2019
    "A Ética das Hienas": "https://letterboxd.com/film/a-etica-das-hienas/",  # Ano: 2019
    "Invasão Espacial": "https://letterboxd.com/film/invasao-espacial/",  # Ano: 2019
    "Teoria Sobre um Planeta Estranho": "https://letterboxd.com/film/teoria-sobre-um-planeta-estranho/",  # Ano: 2018
    "Quanto Pesa": "https://letterboxd.com/film/quanto-pesa/",  # Ano: 2020
    "O Elemento Tinta": "https://letterboxd.com/film/o-elemento-tinta/",  # Ano: 2022
    "Serrão": "https://letterboxd.com/film/serrao/",  # Ano: 2021
    "Um Tempo Para Mim": "https://letterboxd.com/film/um-tempo-para-mim/",  # Ano: 2022
    "Cidade; Campo": "Cidade; Campo",  # Ano: 2024
    "Introdução à Música do Sangue": "https://letterboxd.com/film/introducao-a-musica-do-sangue/",  # Ano: 2015
    "O Fim e os Meios": "https://letterboxd.com/film/o-fim-e-os-meios/",  # Ano: 2015
    "Médico de Monstro": "https://letterboxd.com/film/medico-de-monstro/",  # Ano: 2017
    "A Pedra": "https://letterboxd.com/film/a-pedra/",  # Ano: 2018
    "E o que a gente faz agora?": "https://letterboxd.com/film/e-o-que-a-gente-faz-agora/",  # Ano: 2019
    "O Balido Interno": "https://letterboxd.com/film/o-balido-interno/",  # Ano: 2019
    "Benzedeira": "https://letterboxd.com/film/benzedeira/",  # Ano: 2021
    "Mas Eu Não Sou Alguém": "https://letterboxd.com/film/mas-eu-nao-sou-alguem/",  # Ano: 2021
    "O Fim da Imagem": "https://letterboxd.com/film/o-fim-da-imagem/",  # Ano: 2022
    "Tekoha": "https://letterboxd.com/film/tekoha/",  # Ano: 2022
    "Da Porta Pra Fora": "https://letterboxd.com/film/da-porta-pra-fora/",  # Ano: 2023
    "Luis Fernando Verissimo: O Filme": "https://letterboxd.com/film/luis-fernando-verissimo-o-filme/",  # Ano: 2023
    "Yãmî Yah-Pá: Fim da Noite": "https://letterboxd.com/film/yami-yah-pa/",  # Ano: 2023
    "Jussara": "https://letterboxd.com/film/jussara/",  # Ano: 2023
    "Poemaria": "https://letterboxd.com/film/poemaria/",  # Ano: 2024
    "Toquinho Maravilhoso": "https://letterboxd.com/film/toquinho-maravilhoso/",  # Ano: 2024
    "Mestras": "https://letterboxd.com/film/mestras/",  # Ano: 2024
}

In [5]:

# === EXECUÇÃO COM LOGICA DE ADIÇÃO (APPEND) ===
nome_arquivo = "reviews_completo_com_ano.json"

# 1. Tenta carregar o que já existe para não perder
try:
    with open(nome_arquivo, "r", encoding='utf-8') as f:
        dados_finais = json.load(f)
    print(f"📦 Arquivo existente carregado com {len(dados_finais)} filmes.")
except FileNotFoundError:
    dados_finais = []
    print("🆕 Criando novo arquivo de resultados.")

# 2. Itera sobre os novos links
for titulo, link in filmes_para_linkar.items():
    # Verifica se o filme já não está nos dados_finais para não duplicar
    if any(d['title'] == titulo for d in dados_finais):
        print(f"⏩ Pulando {titulo}, já existe no arquivo.")
        continue
    
    if not link: continue 
    
    criticas = raspar_final(titulo, link) # Aqui você usa o seu filtro de 100 chars
    ano = mapa_anos.get(titulo, "Desconhecido")
    
    dados_finais.append({
        "title": titulo,
        "release_year": ano,
        "reviews": criticas
    })

    # 3. Salva a cada filme processado (Sobrescreve a lista atualizada)
    with open(nome_arquivo, "w", encoding='utf-8') as f:
        json.dump(dados_finais, f, ensure_ascii=False, indent=4)


def raspar_final(titulo, url_base):
    if not url_base: 
        return [] # Pula se você não preencheu o link
    
    # Ajusta URL para aba de reviews
    if "reviews" not in url_base:
        url = f"{url_base}reviews/by/activity/" if url_base.endswith('/') else f"{url_base}/reviews/by/activity/"
    else:
        url = url_base

    reviews = []
    page = 1
    print(f"🎬 Processando: {titulo}...")
    
    while len(reviews) < META_REVIEWS:
        url_page = f"{url}page/{page}/" if page > 1 else url
        try:
            time.sleep(random.uniform(5, 10)) # Delay de respeito
            resp = requests.get(url_page, headers=HEADERS)
            
            if resp.status_code != 200: 
                print(f"   Erro {resp.status_code}")
                break
            
            soup = BeautifulSoup(resp.text, 'html.parser')
            # Busca direta pelo corpo do texto (mais segura contra mudanças de layout)
            bodies = soup.find_all('div', class_='body-text')
            
            if not bodies: 
                print("   Nenhum review encontrado nesta página.")
                break
            
            count = 0
            for b in bodies:
                if len(reviews) >= META_REVIEWS: break
                txt = b.get_text(separator=' ', strip=True)
                
                # Filtros de qualidade
                if len(txt) > 100:
                    try:
                        if detect(txt) == 'pt':
                            reviews.append(txt)
                            count += 1
                    except: pass
            
            print(f"   Pág {page}: {count} reviews novos.")
            if count == 0 and page > 3: break
            page += 1
            
        except Exception as e: 
            print(f"   Erro: {e}")
            break
            
    return reviews

# === EXECUÇÃO ===
dados_finais = []

# Itera sobre o dicionário que você preencheu manualmente
for titulo, link in filmes_para_linkar.items():
    if not link: continue # Ignora os vazios
    
    criticas = raspar_final(titulo, link)
    
    # Recupera o ano do arquivo mestre
    ano = mapa_anos.get(titulo, "Desconhecido")
    
    dados_finais.append({
        "title": titulo,
        "release_year": ano,
        "reviews": criticas
    })

# Salva o arquivo final
nome_arquivo = "reviews_completo.json"
with open(nome_arquivo, "w", encoding='utf-8') as f:
    json.dump(dados_finais, f, ensure_ascii=False, indent=4)

print(f"\n✅ Processo concluído! Salvo em: {nome_arquivo}")

📦 Arquivo existente carregado com 158 filmes.
⏩ Pulando O corpo, já existe no arquivo.
⏩ Pulando O Teto Sobre Nós, já existe no arquivo.
⏩ Pulando Bá, já existe no arquivo.
⏩ Pulando Miss & Grubs, já existe no arquivo.
⏩ Pulando Rosinha, já existe no arquivo.
⏩ Pulando Tailor, já existe no arquivo.
⏩ Pulando O Espirito do Bosque, já existe no arquivo.
⏩ Pulando Telentrega, já existe no arquivo.
⏩ Pulando O Violeiro Fantasma, já existe no arquivo.
⏩ Pulando O Quebra-Cabeça de Sara, já existe no arquivo.
⏩ Pulando Cabelo Bom, já existe no arquivo.
⏩ Pulando Aquarela, já existe no arquivo.
⏩ Pulando Kairo, já existe no arquivo.
⏩ Pulando Menino Pássaro, já existe no arquivo.
⏩ Pulando A Mulher que Sou, já existe no arquivo.
⏩ Pulando O Véu de Amani, já existe no arquivo.
⏩ Pulando A Ética das Hienas, já existe no arquivo.
⏩ Pulando Invasão Espacial, já existe no arquivo.
⏩ Pulando Teoria Sobre um Planeta Estranho, já existe no arquivo.
⏩ Pulando Quanto Pesa, já existe no arquivo.
⏩ Puland

In [8]:
import json

# 1. Carrega os arquivos
with open('reviews_completo_com_ano.json', 'r', encoding='utf-8') as f:
    lista_principal = json.load(f)

with open('reviews_completo.json', 'r', encoding='utf-8') as f:
    lista_novos = json.load(f)

# 2. Dicionário para unificar pelo título
unificados = {}

# Processa a primeira lista
for item in lista_principal + lista_novos:
    titulo = item['title']
    reviews_atuais = item.get('reviews', []) or item.get('critics', [])

    if titulo not in unificados:
        # Se o filme não tá no dicionário, adiciona ele
        unificados[titulo] = item
        # Garante que a chave se chame 'reviews' para padronizar
        unificados[titulo]['reviews'] = reviews_atuais
    else:
        # Se o filme já existe, combina os reviews e remove textos idênticos
        reviews_existentes = unificados[titulo].get('reviews', [])
        # Set remove frases exatamente iguais
        total_reviews = list(set(reviews_existentes + reviews_atuais))
        unificados[titulo]['reviews'] = total_reviews

# 3. Transforma o dicionário de volta em lista
lista_final = list(unificados.values())

# 4. Salva o resultado
with open('base_reviews_final.json', 'w', encoding='utf-8') as f:
    json.dump(lista_final, f, ensure_ascii=False, indent=4)

print(f"✅ Unificação concluída!")
print(f"Filmes totais sem duplicatas: {len(lista_final)}")

✅ Unificação concluída!
Filmes totais sem duplicatas: 158
